# Phase 3 Debug - Mapping Enrichment from Phase 1

Use this notebook to compare the current Phase 3 mapping against the full Phase 1 motor neuron export and generate review tables.

In [ ]:
from pathlib import Path
import json
import sys


def _dedupe_paths(paths):
    seen = set()
    out = []
    for path in paths:
        resolved = path.expanduser().resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        out.append(resolved)
    return out


def _find_phase3_root():
    cwd = Path.cwd().resolve()
    direct = [cwd, *cwd.parents]
    extra = []
    for base in direct:
        extra.extend([base / 'Phase 3_WORKING', base / 'Phase 3'])
    for candidate in _dedupe_paths(direct + extra):
        if (candidate / 'src' / 'phase3_bridge').exists() and (candidate / 'configs' / 'phase3_video_profiles.yaml').exists():
            return candidate
    raise FileNotFoundError('Could not locate the Phase 3 root.')


PHASE3_ROOT = _find_phase3_root()
WORKSPACE_ROOT = next((candidate for candidate in [PHASE3_ROOT.parent, *PHASE3_ROOT.parents] if (candidate / 'Phase 1').exists()), PHASE3_ROOT.parent)
SRC_DIR = PHASE3_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

try:
    import pandas as pd
    from phase3_bridge.mapping_enrichment import run_mapping_enrichment
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        f"Missing dependency {exc.name!r}. Install the Phase 3 requirements first with: pip install -r {PHASE3_ROOT / 'requirements.txt'}"
    ) from exc

PHASE1_MN_EXPORT_CSV = (WORKSPACE_ROOT / 'Phase 1' / 'motor_neuron_query' / 'outputs' / 'all_motor_neurons_instance_contains_MN.csv').resolve()
MAPPING_CSV = (PHASE3_ROOT / 'data' / 'inputs' / 'mappings' / 'mn_to_actuator_mapping_with_groups.csv').resolve()
OUT_DIR = (PHASE3_ROOT / 'data' / 'derived' / 'mapping_enrichment').resolve()
MJCF_CANDIDATES = [
    PHASE3_ROOT / 'data' / 'inputs' / 'flybody' / 'floor.xml',
    Path.home() / 'Desktop' / 'Digifly' / 'flybody-main' / 'flybody' / 'fruitfly' / 'assets' / 'floor.xml',
]
MJCF_XML = next((path.expanduser().resolve() for path in MJCF_CANDIDATES if path.expanduser().exists()), None)
LOW_SCORE_THRESHOLD = 4.0

print('PHASE1_MN_EXPORT_CSV =', PHASE1_MN_EXPORT_CSV)
print('MAPPING_CSV          =', MAPPING_CSV)
print('MJCF_XML             =', MJCF_XML)
print('OUT_DIR              =', OUT_DIR)

In [ ]:
paths = run_mapping_enrichment(
    phase1_mn_csv=PHASE1_MN_EXPORT_CSV,
    mapping_csv=MAPPING_CSV,
    out_dir=OUT_DIR,
    mjcf_xml=MJCF_XML,
    low_score_threshold=LOW_SCORE_THRESHOLD,
)

print('[enrichment] outputs:')
for key, value in paths.items():
    print(f'  {key}: {value}')

summary_path = Path(paths['summary_json'])
summary = json.loads(summary_path.read_text())
summary

In [ ]:
coverage = pd.read_csv(paths['mapping_phase1_coverage_csv'])
missing = pd.read_csv(paths['mapping_missing_template_csv'])
low_conf = pd.read_csv(paths['mapping_low_confidence_csv'])

print('coverage rows =', len(coverage))
print('missing rows  =', len(missing))
print('low-conf rows =', len(low_conf))

coverage.head(20)

In [ ]:
to_fix = missing.copy()
keep_cols = [
    column for column in ['mn_id', 'instance', 'type', 'class', 'side', 'suggested_actuator_name', 'suggestion_basis', 'review_reasons']
    if column in to_fix.columns
]
to_fix = to_fix[keep_cols].sort_values(['suggestion_basis', 'mn_id'], na_position='last')
to_fix.head(40)